# 5. Durable HITL interrupt, resume, and idempotent writes

In [ ]:
print("Embedded dataset and deterministic offline lesson are ready.")

EDUCATIONAL — SELF-CONTAINED

A durable human-in-the-loop (HITL) gate pauses before a simulated write. LangGraph's `interrupt` persists the pending decision in a checkpointer; `Command(resume=...)` continues the same `thread_id`. The write uses an idempotency key, so replay returns one logical receipt.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class HitlState(TypedDict, total=False):
    proposal: str
    idempotency_key: str
    decision: dict
    receipt: dict
    status: str


receipts: dict[str, dict] = {}

def write_once(key: str, action: str) -> dict:
    if key not in receipts:
        receipts[key] = {"receipt_id": "R-001", "action": action, "key": key}
    return receipts[key]


def review_node(state: HitlState) -> dict:
    decision = interrupt({"proposal": state["proposal"], "risk": "inventory hold"})
    return {"decision": decision, "status": "approved" if decision.get("approved") else "rejected"}


def apply_node(state: HitlState) -> dict:
    if state["status"] != "approved":
        return {"receipt": {"status": "not written"}}
    receipt = write_once(state["idempotency_key"], state["proposal"])
    return {"receipt": receipt}


builder = StateGraph(HitlState)
builder.add_node("review", review_node)
builder.add_node("apply", apply_node)
builder.add_edge(START, "review")
builder.add_edge("review", "apply")
builder.add_edge("apply", END)
app = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "case-H-1230-2026"}}
paused = app.invoke({"proposal": "hold SKU-1 at DC-West", "idempotency_key": "hold-case-1"}, config)
print("paused interrupt ->", paused["__interrupt__"][0].value)
assert paused["__interrupt__"][0].value["risk"] == "inventory hold"
resumed = app.invoke(Command(resume={"approved": True, "actor": "food-safety-manager"}), config)
print("resumed state ->", resumed)
first = resumed["receipt"]
second = write_once("hold-case-1", "hold SKU-1 at DC-West")
print("replayed receipt ->", second)
assert resumed["status"] == "approved"
assert first == second and first["receipt_id"] == "R-001"
print("ASSERTION PASSED: interrupt paused, resume preserved thread state, and write replay was idempotent")


In [ ]:
print("EDUCATIONAL — SELF-CONTAINED")
print("Durability means a review can resume; idempotency means retrying does not duplicate the side effect.")
assert list(receipts) == ["hold-case-1"]
print("ASSERTION PASSED: EDUCATIONAL — SELF-CONTAINED")